In [0]:
dbutils.widgets.text("env", "dev")
env     = dbutils.widgets.get("env")
BR, SV  = f"{env}_bronze", f"{env}_silver"
regions = ["africa", "america", "asia", "europe", "middle_east"]
def load(target, select_sql):
    spark.sql(f"INSERT OVERWRITE {target} {select_sql}")
    print(f"✔ {target}: {spark.table(target).count():,} rows")


In [0]:
load(f"{SV}.common.regions", f"""
  SELECT r_regionkey, upper(trim(r_name))
  FROM {BR}.common.region WHERE r_regionkey IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY r_regionkey ORDER BY _ingested_at DESC) = 1""")
load(f"{SV}.common.nations", f"""
  SELECT n_nationkey, upper(trim(n_name)), n_regionkey
  FROM {BR}.common.nation WHERE n_nationkey IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY n_nationkey ORDER BY _ingested_at DESC) = 1""")
load(f"{SV}.common.products", f"""
  SELECT p_partkey, trim(p_name), trim(p_mfgr), trim(p_brand), upper(trim(p_type)),
         p_size, upper(trim(p_container)), p_retailprice
  FROM {BR}.common.part WHERE p_partkey IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY p_partkey ORDER BY _ingested_at DESC) = 1""")
load(f"{SV}.common.suppliers", f"""
  SELECT s_suppkey, trim(s_name), trim(s_address), s_nationkey, trim(s_phone), s_acctbal
  FROM {BR}.common.supplier WHERE s_suppkey IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY s_suppkey ORDER BY _ingested_at DESC) = 1""")
load(f"{SV}.common.product_suppliers", f"""
  SELECT ps_partkey, ps_suppkey, ps_availqty, ps_supplycost
  FROM {BR}.common.partsupp WHERE ps_partkey IS NOT NULL AND ps_suppkey IS NOT NULL
  QUALIFY row_number() OVER (PARTITION BY ps_partkey, ps_suppkey ORDER BY _ingested_at DESC) = 1""")


In [0]:
for r in regions:
    load(f"{SV}.{r}.customers", f"""
      SELECT c_custkey, trim(c_name), trim(c_address), c_nationkey, trim(c_phone),
             c_acctbal, upper(trim(c_mktsegment))
      FROM {BR}.{r}.customer WHERE c_custkey IS NOT NULL
      QUALIFY row_number() OVER (PARTITION BY c_custkey ORDER BY _ingested_at DESC) = 1""")
    load(f"{SV}.{r}.orders", f"""
      SELECT o_orderkey, o_custkey, o_orderstatus,
             CASE o_orderstatus WHEN 'F' THEN 'Fulfilled'
                                WHEN 'O' THEN 'Open'
                                WHEN 'P' THEN 'Partially fulfilled' END,
             o_totalprice, o_orderdate, trim(o_orderpriority), trim(o_clerk), o_shippriority
      FROM {BR}.{r}.orders WHERE o_orderkey IS NOT NULL
      QUALIFY row_number() OVER (PARTITION BY o_orderkey ORDER BY _ingested_at DESC) = 1""")
    load(f"{SV}.{r}.order_lines", f"""
      SELECT l_orderkey, l_linenumber, l_partkey, l_suppkey,
             l_quantity, l_extendedprice, l_discount, l_tax,
             l_returnflag, l_linestatus, l_shipdate, l_commitdate, l_receiptdate,
             upper(trim(l_shipinstruct)), upper(trim(l_shipmode))
      FROM {BR}.{r}.lineitem WHERE l_orderkey IS NOT NULL
      QUALIFY row_number() OVER (PARTITION BY l_orderkey, l_linenumber ORDER BY _ingested_at DESC) = 1""")
